# Avance 5 - Modelo final de la permeabilidad

**Bioactive AI** · Equipo 7 · Maestria en Inteligencia Artificial Aplicada (TC5035.10)

Combina modelos para clasificar la permeabilidad de la barrera hematoencefalica (BBB+ / BBB-), la fase 1 del proyecto. Parte de los modelos individuales del [Avance 4](04_Models.ipynb), arma cinco ensambles que cubren las dos estrategias -**homogeneos**, que combinan copias de una misma familia, y **heterogeneos**, que combinan familias distintas-, los compara con el mejor individual en la misma corrida y elige el modelo final. Al final, y una sola vez, lo evalua en la particion de prueba que ningun cuaderno anterior toco.

La fase 2, la potencia contra cada diana, va en el 05b.

Usa los mismos datos, la misma particion agrupada por andamio y esqueleto de conectividad, las mismas tres semillas y los mismos cinco pliegues que el 03 y el 04, asi que sus cifras se comparan con las de esos cuadernos. Entra **solo con descriptores**: el [03c](03c_Permeabilidad_huellas.ipynb) midio que las huellas Sort & Slice no mejoran a ningun modelo en esta fase.

## 1. Preparacion

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import json
import platform
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"
DATOS = RAIZ / "notebooks" / "datos"

from src.evaluation.cv import _medir, evaluar, intervalo_por_grupos, tabla_comparativa
from src.evaluation.graficos import (curva_precision_exhaustividad, curva_roc,
                                     importancias_dos_metodos, matriz_confusion)
from src.models import cargar, seleccion
from src.models.ajuste import BusquedaAnidada
from src.models.ensambles import ensambles
from src.tracking import tomar_prueba

# Versiones con las que se ejecuto: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

### Los datos

Entrenamiento y validacion de la fase 1, con la prueba reservada fuera. Cada compuesto lleva su **origen** -natural exacto, analogo estereoquimico o sin equivalente natural-, que es el desglose de la decision [1.6](../docs/decisiones.md#d1-6): un promedio de todos no dice como le va al modelo con los productos naturales, que son los que el proyecto quiere priorizar.

In [ ]:
t = cargar("permeabilidad", estratos="origen")
X, y, g, origen = t.X, t.y, t.grupos, t.estratos
print(f"{len(y):,} compuestos · {X.shape[1]} descriptores · {y.mean():.1%} BBB+ · "
      f"{len(np.unique(g)):,} grupos")
print(pd.Series(origen).value_counts().to_string())

## 2. Los modelos

### Los individuales de referencia

Los cuatro mejores del Avance 4 por MCC, sin afinar: el SVM con nucleo RBF, la regresion logistica, la red de una capa y los vecinos mas cercanos. El SVM fue el individual que eligio el 04 y es **el mejor individual** contra el que se mide cada ensamble.

**Se vuelven a correr aqui, y las cifras del 04 no se usan como referencia.** La comparacion tiene que salir de una sola corrida -mismos pliegues, mismas semillas y misma maquina- para que la columna de tiempos signifique lo mismo en todas las filas. Y hay una razon mas fuerte: las cifras registradas del 04 (el SVM en 0.703 de MCC) se midieron sobre otra matriz de descriptores. La misma version de RDKit calcula flotantes distintos segun como se compilo, asi que la matriz de la fase 1 no es identica entre las maquinas del equipo ([#166](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/166)), y cual se declara canonica esta por decidir. Lo que no depende de esa decision es la comparacion de aqui: **todos los modelos de esta corrida, individuales y ensambles, leen la misma matriz**, y la diferencia pareada contra el SVM medido aqui es limpia aunque la matriz cambie despues. `tracking.py` anota el sha256 de lo que leyo, asi que siempre se sabra con cual se midio.

Esos cuatro son tambien la base de la votacion y del stacking, como pide la rubrica.

### Los ensambles

| Ensamble | Estrategia | Que combina |
|:--|:--|:--|
| `bosque` | homogenea, bagging de arboles con columnas al azar | 300 arboles de decision |
| `boosting` | homogenea, secuencial: cada arbol corrige lo que fallo el anterior | arboles poco profundos |
| `bagging_svm` | homogenea, bagging | 10 copias del SVM RBF, cada una sobre un remuestreo de las filas |
| `votacion` | heterogenea, promedio de las probabilidades | los cuatro individuales |
| `stacking` | heterogenea, una regresion logistica aprende a combinarlos | los cuatro individuales |

El bosque y el boosting son los del baseline del 03, sin cambios, para que sus filas se puedan contrastar con lo que ya esta registrado; ademas van sus dos versiones afinadas.

**El stacking es el unico que puede filtrar sin que se note.** Su meta-modelo aprende de predicciones que los modelos base hacen fuera de pliegue, y esos pliegues internos se arman al azar por omision: con analogos del mismo esqueleto a los dos lados, las predicciones de entrenamiento salen mejores de lo que seran en validacion y el meta-modelo aprende a confiar de mas en el que memoriza. El ajuste externo seguiria viendose honesto. `StackingAgrupado` parte con los grupos del pliegue externo, por la misma razon que `BusquedaAnidada` ([1.2](../docs/decisiones.md#d1-2)).

**Lo que se ejecuta es el codigo de `src/`**, que importa la celda de arriba. Se muestra aqui para leerlo sin abrir otro archivo, y `tests/test_05_codigo.py` comprueba que lo mostrado y lo importado sean identicos.

```python
class StackingAgrupado(BaseEstimator, ClassifierMixin):
    """Stacking cuya validación interna agrupa con los grupos del pliegue externo.

    `estimadores` es una lista de `(nombre, canalización)`. El meta-modelo recibe la
    probabilidad de la clase positiva de cada uno, calculada fuera de los pliegues
    internos, y por defecto es una regresión logística: con tantas entradas como
    modelos base, y correlacionadas, un meta-modelo flexible sobreajusta.
    """

    necesita_grupos = True

    def __init__(self, estimadores: Sequence[tuple[str, BaseEstimator]],
                 final: BaseEstimator | None = None, *, pliegues: int = 3,
                 semilla: int = 0):
        self.estimadores = estimadores
        self.final = final
        self.pliegues = pliegues
        self.semilla = semilla

    def fit(self, X, y, grupos=None):
        if grupos is None:
            raise ValueError(
                "StackingAgrupado necesita `grupos`: sin ellos el meta-modelo aprendería "
                "de predicciones hechas con análogos del mismo esqueleto a los dos lados. "
                "El arnés los pasa solo; fuera de él, llame fit(X, y, grupos=...).")
        y = np.asarray(y)
        interno = StratifiedGroupKFold(n_splits=self.pliegues, shuffle=True,
                                       random_state=self.semilla)
        particiones = list(interno.split(X, y, np.asarray(grupos)))
        final = LogisticRegression(max_iter=3000) if self.final is None else clone(self.final)
        self.stacking_ = StackingClassifier(
            [(n, clone(e)) for n, e in self.estimadores], final_estimator=final,
            cv=particiones, stack_method="predict_proba").fit(X, y)
        self.classes_ = self.stacking_.classes_
        return self

    def predict(self, X):
        return self.stacking_.predict(X)

    def predict_proba(self, X):
        return self.stacking_.predict_proba(X)

def ensambles(base: Sequence[str], semilla: int = 0) -> dict[str, BaseEstimator]:
    """Los cinco ensambles, con `base` como los individuales de la votación y el stacking.

    `base` son claves del catálogo de `alternativos.py`; el bagging copia el SVM RBF
    porque es el mejor individual del Avance 4. El bosque y el boosting son los del
    baseline del 03, sin cambios, para que su fila se pueda contrastar con la que ya
    está registrada.
    """
    individuales = seleccion(tuple(base), semilla)
    referencia = canalizaciones(semilla)
    # El bosque del baseline usa todos los núcleos; aquí uno solo, como los demás, para que
    # la columna de tiempos compare modelos y no paralelismo. No cambia sus predicciones.
    referencia["bosque"].set_params(modelo__n_jobs=1)
    svm = seleccion(("svm_rbf",), semilla)["svm_rbf"]
    return {
        "bosque": referencia["bosque"],
        "boosting": referencia["boosting"],
        "bagging_svm": Pipeline([
            ("modelo", BaggingClassifier(svm, n_estimators=10, random_state=semilla))]),
        "votacion": Pipeline([
            ("modelo", VotingClassifier(list(individuales.items()), voting="soft"))]),
        "stacking": StackingAgrupado(list(individuales.items()), semilla=semilla),
    }
```

### El ajuste de hiperparametros

La rubrica pide optimizar los modelos relevantes. Se afinan el bosque y el boosting, que son los dos ensambles homogeneos con hiperparametros que cambian su comportamiento; el bagging del SVM hereda los del individual que ya eligio el 04, y la votacion y el stacking no tienen propios mas alla de sus bases.

Las dos rejillas tienen 18 combinaciones y se recorren completas, con tres pliegues internos **tambien agrupados**: la busqueda vive dentro de cada pliegue externo, asi que la validacion del arnes nunca participa en elegir. La metrica que elige es el MCC, la principal de la fase 1 ([C.1](../docs/decisiones.md#dc-1)). La columna `elegidos` del resultado guarda que eligio en cada pliegue, que es lo que permite decir si la eleccion fue estable o cambio con la particion.

| Modelo | Parametro | Valores | Que controla |
|:--|:--|:--|:--|
| `bosque` | `max_features` | `sqrt`, 0.3, 0.5 | cuantas columnas ve cada division, que es lo que diversifica los arboles |
| | `min_samples_leaf` | 1, 2, 5 | cuanto puede memorizar cada hoja |
| | `max_depth` | sin limite, 20 | un freno adicional al sobreajuste |
| `boosting` | `learning_rate` | 0.03, 0.1, 0.3 | cuanto corrige cada arbol |
| | `max_leaf_nodes` | 15, 31, 63 | la complejidad de cada arbol |
| | `l2_regularization` | 0, 1 | la penalizacion de las hojas |

**No se usa la parada temprana del boosting**, que separa su propia validacion al azar y sin agrupar: volveria a meter la fuga que todo este arnes evita.

In [ ]:
SEMILLAS = (0, 1, 2)
BASE = ("svm_rbf", "logistica", "mlp", "knn")     # los cuatro mejores del 04 por MCC
MEJOR_INDIVIDUAL = "svm_rbf"                     # el que eligio el 04; contra el se mide cada ensamble
MINIMO_PLIEGUES = 12                             # de 15 evaluaciones, en cuantas tiene que mejorar
PRINCIPALES = ("mcc", "auprc_minoritaria")       # las que deciden; el AUROC es tercera lectura
MUESTRA_ENTRENAMIENTO = 3000                     # filas de entrenamiento para medir la brecha

REJILLAS = {
    "bosque": {"modelo__max_features": ["sqrt", 0.3, 0.5],
               "modelo__min_samples_leaf": [1, 2, 5],
               "modelo__max_depth": [None, 20]},
    "boosting": {"modelo__learning_rate": [0.03, 0.1, 0.3],
                 "modelo__max_leaf_nodes": [15, 31, 63],
                 "modelo__l2_regularization": [0.0, 1.0]},
}

individuales = seleccion(BASE)
MODELOS = {**individuales, **ensambles(BASE)}
MODELOS.update({f"{n} afinado": BusquedaAnidada(MODELOS[n], r, pliegues=3)
                for n, r in REJILLAS.items()})
ORDEN = list(MODELOS)
ENSAMBLES = [n for n in ORDEN if n not in individuales]
print(f"{len(individuales)} individuales y {len(ENSAMBLES)} ensambles")
print(" · ".join(ENSAMBLES))

### Lo que se fija antes de ver los resultados

- **Las metricas principales son el MCC y el AUPRC de la clase minoritaria** ([C.1](../docs/decisiones.md#dc-1)); el AUROC se reporta como tercera lectura y no decide. La tabla se ordena por MCC.
- **Un ensamble mejora al mejor individual solo si** la diferencia media supera el umbral **y** es positiva en al menos **12 de las 15** evaluaciones, en las dos metricas principales a la vez. Las diferencias se calculan contra `svm_rbf` **en la misma evaluacion** (misma semilla, mismo pliegue), no entre promedios. Es la misma regla con la que se leyeron el 04c y el 03c ([1.3](../docs/decisiones.md#d1-3)), y se usa aqui por consistencia.
- **El umbral de cada metrica es la mayor desviacion de esa metrica entre las 15 evaluaciones del mejor individual**, calculado con ese modelo solo, antes de mirar los ensambles. La regla **no esta pre-registrada**: se fijo despues de ver el control de los cuadernos anteriores. En la fase 1 el MCC varia mucho entre pliegues, asi que el umbral sale alto y la regla es conservadora; si un ensamble gana en 15 de 15 con una diferencia estable por debajo del umbral, se reporta asi y el umbral no se baja.
- **Los 15 pares no son independientes**: las semillas comparten filas y los pliegues parten los mismos datos. Los 12 de 15 son una prueba de signos con esa limitacion, no una probabilidad.
- **Ordenar la tabla no declara un ganador.** La desviacion entre evaluaciones es dispersion y no error estandar. Para afirmar que dos modelos se distinguen se remuestrean **grupos** con `intervalo_por_grupos`. El Avance 3 cerro con que ninguno de sus tres modelos se distinguia, y si eso se repite aqui, decirlo es el resultado.
- **Los desgloses por origen son exploratorios.** La comparacion que decide es la del agregado.
- **La particion de prueba se toca una sola vez**, al final, con el modelo final ya elegido, a traves de `tomar_prueba(final=True, motivo=...)`, que deja registro en `experimentos/prueba.csv` ([#54](https://github.com/AntonioVazquezMartinez/Bioactive-AI/issues/54)).

## 3. La corrida

In [ ]:
inicio = time.perf_counter()
res, fdp = evaluar(MODELOS, X, y, g, estratos=origen, semillas=SEMILLAS, verboso=False)
print(f"{len(res):,} filas · {(time.perf_counter() - inicio) / 60:.1f} min")

### La tabla comparativa

Una fila por modelo, ordenada por MCC, con el AUPRC de la minoritaria, el AUROC y el tiempo medio de ajuste por pliegue. `desv` es la dispersion del MCC entre las 15 evaluaciones, no un error estandar.

In [ ]:
tabla = tabla_comparativa(res, principal="mcc")
tabla

### Que eligio la busqueda en cada pliegue

Si los hiperparametros elegidos cambian de un pliegue a otro, el afinado no esta encontrando un optimo estable sino acomodandose a cada particion, y esa es la lectura que importa, no la cifra.

In [ ]:
elegidos = (res[res["elegidos"].notna() & res["estrato"].eq("todos")]
            .groupby("modelo")["elegidos"].value_counts().rename("evaluaciones"))
elegidos.to_frame()

### Los ensambles contra el mejor individual

Para cada ensamble se resta el MCC y el AUPRC del `svm_rbf` en **la misma evaluacion**. `mejor_en` cuenta en cuantas de las 15 gana, y `mejora` aplica la regla fijada arriba. Los umbrales se calculan primero, solo con el mejor individual.

In [ ]:
solo = res[res["estrato"].eq("todos")]
UMBRALES = {m: round(float(solo[solo["modelo"].eq(MEJOR_INDIVIDUAL)][m].std()), 3)
            for m in PRINCIPALES}
print(f"Umbrales, la desviacion de {MEJOR_INDIVIDUAL} entre sus 15 evaluaciones: {UMBRALES}")


def comparacion(r, estrato="todos"):
    """Diferencia de cada modelo contra el mejor individual, evaluacion por evaluacion."""
    r = r[r["estrato"] == estrato]
    filas = []
    for metrica in ("mcc", "auprc_minoritaria", "auroc"):
        p = r.pivot_table(index=["modelo", "semilla", "pliegue"], values=metrica).unstack("modelo")[metrica]
        for modelo in [m for m in ORDEN if m != MEJOR_INDIVIDUAL]:
            dif = (p[modelo] - p[MEJOR_INDIVIDUAL]).dropna()
            filas.append({"modelo": modelo, "metrica": metrica, "evaluaciones": len(dif),
                          "dif_media": dif.mean(), "dif_desv": dif.std(),
                          "mejor_en": int((dif > 0).sum())})
    c = pd.DataFrame(filas).set_index(["modelo", "metrica"])
    c["cumple"] = (c["dif_media"].gt(c.index.get_level_values("metrica").map(UMBRALES).astype(float))
                   & c["mejor_en"].ge(MINIMO_PLIEGUES))
    mejora = c["cumple"].unstack("metrica")[list(PRINCIPALES)].all(axis=1).rename("mejora")
    return c.round(4), mejora


comparacion_todos, mejora_todos = comparacion(res)
display(comparacion_todos.unstack("metrica")[["dif_media", "mejor_en"]].round(3))
mejora_todos.to_frame()

### Si dos modelos se distinguen o no

La tabla ordena por la media, y eso no prueba que la primera fila se distinga de la segunda. Aqui se remuestrean **grupos** -no filas, porque la fila no es la unidad independiente- para el mejor individual y los dos primeros ensambles de la tabla. Si los intervalos se solapan, ninguno gana y eso es lo que hay que decir.

In [ ]:
candidatos = [MEJOR_INDIVIDUAL] + [m for m in tabla.index if m in ENSAMBLES][:2]
filas = []
for m in candidatos:
    auroc, lo, hi = intervalo_por_grupos(y, fdp[m][0], g)   # remuestrea grupos, no filas
    filas.append({"modelo": m, "auroc": auroc, "inferior": lo, "superior": hi})
intervalos = pd.DataFrame(filas).set_index("modelo").round(4)
intervalos

### Por origen (exploratorio, no decide)

Un promedio puede esconder como le va a los productos naturales, que son los que el proyecto quiere priorizar. El [03c](03c_Permeabilidad_huellas.ipynb) y el [04c](04c_Regresion_actividad_huellas.ipynb) encontraron que las huellas solo ayudaban en los compuestos sin equivalente natural; aqui se mira si los ensambles reparten su ganancia igual o distinto.

In [ ]:
por_origen = pd.unique(res.loc[res["estrato"].ne("todos"), "estrato"])
mcc_origen = pd.concat({e: res[res["estrato"].eq(e)].groupby("modelo")["mcc"].mean().loc[ORDEN]
                        for e in por_origen}, axis=1).round(4)
display(mcc_origen)
pd.concat({e: comparacion(res, e)[0].query("metrica in @PRINCIPALES")[["dif_media", "mejor_en", "cumple"]]
           for e in por_origen}, names=["origen"])

### La brecha entre entrenamiento y validacion

Un modelo que se aprende los ejemplos acierta mucho en lo que vio y mucho menos en lo que no. La brecha es el MCC sobre una muestra de 3,000 filas de entrenamiento menos el de validacion, con la semilla 0 y sus cinco pliegues. Los ensambles de arboles ya sobreajustaban en el 03, asi que se compara cada modelo contra los demas sabiendo eso. Con el `knn` la muestra de entrenamiento incluye a cada punto como su propio vecino, asi que su brecha sale inflada y solo se compara contra si misma.

In [ ]:
filas = []
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)
for k, (ajuste, val) in enumerate(cv.split(X, y, g), 1):
    muestra = np.random.default_rng(k).choice(ajuste, min(MUESTRA_ENTRENAMIENTO, len(ajuste)),
                                              replace=False)
    for nombre, pipe in MODELOS.items():
        m = clone(pipe)
        if getattr(pipe, "necesita_grupos", False):
            m.fit(X[ajuste], y[ajuste], grupos=g[ajuste])
        else:
            m.fit(X[ajuste], y[ajuste])
        filas.append({"modelo": nombre, "pliegue": k,
                      "mcc_entrenamiento": _medir(y[muestra], m.predict_proba(X[muestra])[:, 1])["mcc"],
                      "mcc_validacion": _medir(y[val], m.predict_proba(X[val])[:, 1])["mcc"]})
brecha = pd.DataFrame(filas).groupby("modelo")[["mcc_entrenamiento", "mcc_validacion"]].mean()
brecha["brecha"] = brecha["mcc_entrenamiento"] - brecha["mcc_validacion"]
brecha.loc[ORDEN].round(4)

In [ ]:
# Registro de la comparativa en experimentos/ (decision 1.5: que modelo y con que representacion).
from src.tracking import Corrida

with Corrida("ensambles-permeabilidad", decision="1.5",
             params={"individuales": list(individuales), "ensambles": ENSAMBLES,
                     "base_heterogeneos": list(BASE), "mejor_individual": MEJOR_INDIVIDUAL,
                     "pliegues": 5, "representacion": "descriptores",
                     "metricas_principales": list(PRINCIPALES),
                     "metrica_tercera_lectura": "auroc",
                     "rejillas": {n: {k: list(map(str, v)) for k, v in r.items()}
                                  for n, r in REJILLAS.items()},
                     "pliegues_internos_busqueda": 3,
                     "regla_mejora": {"umbral_por_metrica": UMBRALES,
                                      "minimo_evaluaciones": MINIMO_PLIEGUES, "de": 15,
                                      "criterio": "las dos metricas principales a la vez",
                                      "umbral": "mayor desviacion del mejor individual entre sus 15 evaluaciones",
                                      "comparacion_decisiva": f"agregado contra {MEJOR_INDIVIDUAL}",
                                      "estratos": "exploratorios",
                                      "estado": "no pre-registrada; la misma del 04c y el 03c"},
                     "estratos": "origen"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "bbb_meta.parquet",
                    PROCESADOS / "bbb_descriptores_filtrados_sin_normalizar.parquet"]) as c:
    c.metricas({"comparativa": tabla.to_dict(orient="index"),
                "comparacion_todos": {f"{m}/{k}": v for (m, k), v in
                                      comparacion_todos.to_dict(orient="index").items()},
                "intervalos": intervalos.to_dict(orient="index"),
                "brecha": brecha.to_dict(orient="index"),
                **{e: res[res["estrato"].eq(e)].groupby("modelo")[["mcc", "auprc_minoritaria", "auroc"]]
                   .mean().to_dict(orient="index") for e in por_origen}})

print(c.carpeta.name)

In [ ]:
# Las 15 evaluaciones por modelo y estrato. El cuaderno lo **escribe** y no lo lee: las
# conclusiones salen de las tablas de arriba. Va despues del registro a proposito, porque
# escribirlo antes dejaria el arbol sucio al registrar.
SALIDA = DATOS / "05_ensambles.csv"
res.to_csv(SALIDA, index=False)
print(f"{SALIDA.relative_to(RAIZ)} · {len(res):,} filas")

## 4. El modelo final

*La eleccion se escribe aqui despues de leer las tablas, y no la decide la primera fila: la rubrica pide argumentos mas alla de la metrica. Lo que hay que pesar, dicho antes de ver los resultados para que la decision no se acomode a ellos: si los intervalos se solapan, cuanto cuesta cada ajuste, cuanto se puede explicar del modelo -la [C.2](../docs/decisiones.md#dc-2) pide un control interpretable al lado-, y como le va con los productos naturales, que es el estrato que el proyecto quiere priorizar.*

*Por completar por el equipo.*

In [ ]:
FINAL = ""                                       # la clave del modelo elegido, de ORDEN
MOTIVO = ""                                      # por que, en una linea, para la bitacora de la prueba
assert FINAL in MODELOS and MOTIVO, "escribe FINAL y MOTIVO antes de seguir"

## 5. Los cuatro graficos

Los cuatro se dibujan con las **predicciones fuera de pliegue** del modelo final -las que devuelve el arnes, una por compuesto, hecha por un modelo que no lo vio al ajustar- y no con un modelo reajustado sobre todo: un grafico hecho sobre datos que el modelo vio describe su memoria y no su desempeno. Las funciones estan en `src/evaluation/graficos.py` y **dibujan, no interpretan**: la interpretacion de cada uno la escribe el equipo abajo.

In [ ]:
p_fdp = fdp[FINAL][0]                            # semilla 0, una prediccion por compuesto
fig, ejes = plt.subplots(2, 2, figsize=(11, 9))
curva_roc(y, p_fdp, ax=ejes[0, 0], etiqueta=FINAL)
ejes[0, 0].set_title("1. Curva ROC")
curva_precision_exhaustividad(y, p_fdp, ax=ejes[0, 1], etiqueta="BBB- (minoritaria)")
ejes[0, 1].set_title("2. Precision-exhaustividad de la minoritaria")
matriz_confusion(y, p_fdp, ax=ejes[1, 0], nombres=("BBB-", "BBB+"))
ejes[1, 0].set_title("3. Matriz de confusion")
plt.tight_layout()

**Interpretacion de los tres primeros.** *Por completar por el equipo, mirando las figuras.*

### El cuarto: que caracteristicas sostienen la prediccion

Dos metodos independientes, como exige la [C.2](../docs/decisiones.md#dc-2): la caida de desempeno al permutar una columna, medida sobre el pliegue de evaluacion, y la importancia por impureza del bosque, que es barata y se sesga hacia variables de muchos valores. Se reporta **solo lo que los dos coinciden en poner arriba**, que es la parte defendible; la barra en azul fuerte marca esa interseccion.

La permutacion se mide sobre el modelo final y la impureza sobre el bosque, porque no todos los modelos tienen una importancia interna. Cualquier afirmacion estructural que salga de aqui hay que contrastarla contra TPSA (< 67 A2) y los donadores de hidrogeno (<= 1) antes de presentarla, porque esas dos propiedades dominan empiricamente la permeabilidad.

In [ ]:
filas = []
cv = StratifiedGroupKFold(5, shuffle=True, random_state=0)
bosque = clone(MODELOS["bosque"])
for k, (ajuste, val) in enumerate(cv.split(X, y, g), 1):
    m = clone(MODELOS[FINAL])
    if getattr(MODELOS[FINAL], "necesita_grupos", False):
        m.fit(X[ajuste], y[ajuste], grupos=g[ajuste])
    else:
        m.fit(X[ajuste], y[ajuste])
    perm = permutation_importance(m, X[val], y[val], n_repeats=5, random_state=k,
                                  scoring="matthews_corrcoef")
    imp = bosque.fit(X[ajuste], y[ajuste]).named_steps["modelo"].feature_importances_
    for j, nombre in enumerate(t.meta.attrs.get("columnas", range(X.shape[1]))):
        filas.append({"caracteristica": str(nombre), "pliegue": k,
                      "permutacion": perm.importances_mean[j], "impureza": imp[j]})
importancias = pd.DataFrame(filas)
fig, ax = plt.subplots(figsize=(7, 6))
importancias_dos_metodos(importancias, ax=ax)
ax.set_title("4. Importancia por permutacion, con la interseccion marcada")
plt.tight_layout()

**Interpretacion del cuarto.** *Por completar por el equipo.*

### La prueba reservada, una sola vez

Hasta aqui todo se midio con validacion cruzada sobre entrenamiento y validacion. El objetivo 3.6 pide evaluar en datos no vistos, y esta es **la unica vez** que el proyecto toca la particion de prueba: 1,499 compuestos que ningun cuaderno anterior leyo, con 57.3% BBB+ contra 64.0% del entrenamiento, porque los andamios raros se le asignaron a proposito ([1.2](../docs/decisiones.md#d1-2)).

El modelo final se reajusta con **todo** entrenamiento y validacion -mas datos, mismos hiperparametros- y predice la prueba una vez. `tomar_prueba` exige `final=True` y un motivo, y escribe la fila en `experimentos/prueba.csv`, que va versionado: si alguien la tocara dos veces, se veria en el diff. La llamada va **dentro** de su propio `with Corrida(...)`, porque esa escritura ensucia el arbol y hacerla antes de entrar al contexto registraria `arbol_sucio: true` sin que nadie haya tocado el codigo.

Las cifras de la prueba **no se comparan** con las de la tabla de arriba como si fueran lo mismo: alli son promedios de 15 evaluaciones sobre una distribucion parecida a la de entrenamiento, y aqui es una sola medicion sobre una distribucion distinta a proposito. Se espera que bajen.

In [ ]:
# La prueba se mide dentro de su propia corrida, y `tomar_prueba` se llama **dentro** del
# contexto a proposito: escribe una linea en experimentos/prueba.csv, que va versionado, asi
# que llamarla antes de entrar haria que la corrida registrara `arbol_sucio: true`.
meta = pd.read_parquet(PROCESADOS / "bbb_meta.parquet").assign(fila=lambda d: np.arange(len(d)))
completo = cargar("permeabilidad", estratos="origen", incluir_prueba=True)

with Corrida("modelo-final-permeabilidad", decision="1.5",
             params={"modelo_final": FINAL, "motivo": MOTIVO,
                     "representacion": "descriptores",
                     "ajustado_con": "entrenamiento y validacion, sin la prueba",
                     "prueba": "una sola vez, por tomar_prueba(final=True)",
                     "estratos": "origen"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "bbb_meta.parquet",
                    PROCESADOS / "bbb_descriptores_filtrados_sin_normalizar.parquet"]) as c:
    prueba = tomar_prueba(meta.rename(columns={"origen": "estrato"}), final=True, motivo=MOTIVO)
    idx = prueba["fila"].to_numpy()
    modelo_final = clone(MODELOS[FINAL])
    if getattr(MODELOS[FINAL], "necesita_grupos", False):
        modelo_final.fit(X, y, grupos=g)
    else:
        modelo_final.fit(X, y)
    p_prueba = modelo_final.predict_proba(completo.X[idx])[:, 1]
    y_prueba = completo.y[idx]
    origen_prueba = completo.estratos[idx]
    en_prueba = {"todos": _medir(y_prueba, p_prueba)}
    for e in pd.unique(origen_prueba):
        sel = origen_prueba == e
        ys = y_prueba[sel]
        if min(int(ys.sum()), int((1 - ys).sum())) >= 20:    # el mismo minimo que el arnes
            en_prueba[str(e)] = _medir(ys, p_prueba[sel])
    c.metricas({"prueba": en_prueba, "validacion_cruzada": tabla.loc[FINAL].to_dict()})

print(c.carpeta.name)

### Las cifras de la prueba, en total y por origen

In [ ]:
pd.DataFrame(en_prueba).T.assign(
    n=[len(y_prueba)] + [int((origen_prueba == e).sum()) for e in list(en_prueba)[1:]],
    positivos=[y_prueba.mean()] + [y_prueba[origen_prueba == e].mean() for e in list(en_prueba)[1:]],
).round(4)

# Declaracion de uso de IA y contribucion

### Uso de herramientas de IA

| Herramienta utilizada | Proposito de uso | Alcance del apoyo (secciones) | Responsabilidad: que reviso o valido el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programacion]. https://claude.com/claude-code | Nos apoyo con la generacion de codigo de los ensambles, con la particion agrupada de la validacion interna del stacking y con el armado de la comparativa | Las secciones que nos ayudo a implementar fueron celdas de codigo | La eleccion de los ensambles y de sus bases, la regla de mejora, la eleccion del modelo final, la interpretacion de los cuatro graficos y la lectura de los resultados fueron del equipo |

### Contribucion individual

| Integrante | Contribucion al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | *Por completar* |
| Artemio Santiago Padilla Robles | *Por completar* |
| Jose Antonio Vazquez Martinez | *Por completar* |